# Introductory PyTorch example: MLP approximation of a sine function

Learn the training loop before studying the four scientific notebooks. Requires PyTorch, NumPy and Matplotlib.

In [3]:
"""Introductory MLP: fit sin(2*pi*x) with PyTorch on CPU."""
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(7)
np.random.seed(7)

# Training and validation inputs are distinct.
x_train = torch.linspace(0, 1, 101).unsqueeze(1)
x_val = torch.linspace(0.005, 0.995, 100).unsqueeze(1)
y_train = torch.sin(2 * torch.pi * x_train)
y_val = torch.sin(2 * torch.pi * x_val)
loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)

# One input, one output, and two small hidden layers.
model = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32),
                      nn.Tanh(), nn.Linear(32, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
train_loss, val_loss = [], []

# Training
for epoch in range(200):
    model.train()
    for xb, yb in loader:
        prediction = model(xb)           # Forward: predict y from x.
        loss = loss_fn(prediction, yb)   # Loss: compare to sin(2*pi*x).
        optimizer.zero_grad()           # Clear gradients from previous batch.
        loss.backward()                 # Backward: compute gradients.
        optimizer.step()                # Update the MLP parameters.
    # Validation
    model.eval()
    with torch.no_grad():
        train_loss.append(loss_fn(model(x_train), y_train).item())
        val_loss.append(loss_fn(model(x_val), y_val).item())
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d}: train={train_loss[-1]:.6f}, val={val_loss[-1]:.6f}")

# Inference
model.eval()
with torch.no_grad():
    prediction = model(x_val).numpy().ravel()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].semilogy(range(1, 201), train_loss, label="Training")
axes[0].semilogy(range(1, 201), val_loss, label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Mean squared error", title="Training history")
axes[0].legend()
axes[1].plot(x_val.numpy().ravel(), y_val.numpy().ravel(), label="Exact sine")
axes[1].plot(x_val.numpy().ravel(), prediction, "--", label="MLP prediction")
axes[1].set(xlabel="x", ylabel="y", title="Function approximation")
axes[1].legend()
fig.tight_layout()
output = Path(__file__).with_name("mlp_sine_results.png") if "__file__" in globals() else Path("mlp_sine_results.png")
fig.savefig(output, dpi=160)
print(f"Saved {output}")
plt.show()


Epoch  40: train=0.042242, val=0.038724
Epoch  80: train=0.021348, val=0.020184
Epoch 120: train=0.013212, val=0.012968
Epoch 160: train=0.002561, val=0.002325
Epoch 200: train=0.001147, val=0.001013
Saved mlp_sine_results.png


C:\Users\professional\AppData\Local\Temp\ipykernel_27728\3398579600.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
